In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mrl_trace import paths
from mrl_trace.stats import bootstrap_ci

# QUICK=True re-runs a fast, few-seed version of an experiment in-kernel (serial, no Pool);
# the default (QUICK=False) REPLAYS the committed 20-seed grid so every figure renders
# instantly. Heavy sweeps (exp19) always document a `--full` shell command instead.
QUICK = False

GREEN, INDIGO, RED, GOLD, GREY = "#3aa07a", "#2f4b8f", "#c0392b", "#e0a93b", "#9aa6b2"

def _clean(ax):
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    ax.set_axisbelow(True); ax.grid(True, color="0.88", lw=0.5)

print("data/results:", paths.results_dir())

In [ ]:
if QUICK:
    from mrl_trace.bandit import run_reversal, _summarize_reversal
    conds = ("device", "abstract", "no_trace")
    raw = {c: run_reversal(c, B=6, trials=4000, n_phases=2, tau_leak=10.0) for c in conds}
    r = _summarize_reversal(raw, conds, 4000, 0.75, 0.5)
else:
    r = paths.load_result("exp18_reversal.npy")

conds = ("device", "abstract", "no_trace")
cmap = {"device": GREEN, "abstract": INDIGO, "no_trace": GREY}
flips, crit, chance = r["flips"], r["crit"], r["chance"]
fig, ax = plt.subplots(figsize=(7.2, 3.6))
for c in conds:
    cur = np.asarray(r["curves"][c], float)
    win = max(1, len(cur) // 120)
    sm = np.convolve(cur, np.ones(win) / win, mode="valid")
    ax.plot(np.arange(len(sm)), sm, color=cmap[c], lw=1.6, label=c)
for f in flips:
    ax.axvline(f, ls="--", color=RED, lw=1.0)
ax.axhline(crit, ls=":", color=GREY, lw=1.0); ax.axhline(chance, ls=":", color=GREY, lw=0.8)
ax.set_xlabel("trial"); ax.set_ylabel("reward rate (running)"); ax.set_ylim(0, 1.03)
ax.set_title("Reversal: device unlearns then re-acquires at the flip"); ax.legend(frameon=False, fontsize=8)
_clean(ax); plt.show()

cr = r["criteria"]
print("pre-registered criteria:", {k: ("PASS" if v else "fail") for k, v in cr.items()})
for c in conds:
    pre, post = np.asarray(r["pre"][c]).mean(), np.asarray(r["post"][c]).mean()
    print(f"  {c:9s}: pre {pre:.3f}  post {post:.3f}")
rc = r["reacq_cost"]
print(f"re-acquisition cost: device {rc['device']} vs abstract {rc['abstract']} trials "
      f"({rc['ratio']:.1f}x -- retention<->flexibility trade-off)"
      if rc.get("ratio") == rc.get("ratio") else "re-acquisition cost: n/a")

In [ ]:
r = paths.load_result("exp19_multitimescale.npy")
labels = [l for l in ["hetero_raw", "hetero_homeo", "hetero_oracle", "best_single", "no_trace"]
          if l in r["finals"]]
lab_col = {"hetero_raw": RED, "hetero_homeo": GREEN, "hetero_oracle": INDIGO,
           "best_single": GOLD, "no_trace": GREY}
means = [np.asarray(r["finals"][l]).mean() for l in labels]
cis = [r["ci"][l] if "ci" in r else bootstrap_ci(np.asarray(r["finals"][l])) for l in labels]
err = [[m - lo for m, (lo, hi) in zip(means, cis)], [hi - m for m, (lo, hi) in zip(means, cis)]]
fig, ax = plt.subplots(figsize=(6.6, 3.6))
x = np.arange(len(labels))
ax.bar(x, means, color=[lab_col[l] for l in labels], width=0.62)
ax.errorbar(x, means, yerr=err, fmt="none", ecolor="0.3", capsize=3, lw=1.0)
ax.axhline(r.get("crit", 0.75), ls="--", color=GREY, lw=1.0)
ax.axhline(r.get("chance", 0.5), ls=":", color=GREY, lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=25, ha="right", fontsize=8)
ax.set_ylabel("overall reward rate"); ax.set_ylim(0, 1.03)
ax.set_title("Multi-timescale: homeostasis recovers the naive measured spread"); _clean(ax); plt.show()
if "criteria" in r:
    print("criteria:", {k: ("PASS" if v else "fail") for k, v in r["criteria"].items()})

In [ ]:
r = paths.load_result("exp22_wm_stc.npy")
taus = np.asarray(r["tau_band"], float)
def _mci(vecs):
    m = np.array([np.asarray(vecs[t]).mean() for t in taus])
    lo = np.array([np.percentile(np.asarray(vecs[t]), 2.5) for t in taus])
    hi = np.array([np.percentile(np.asarray(vecs[t]), 97.5) for t in taus])
    return m, lo, hi
fig, (axA, axB) = plt.subplots(1, 2, figsize=(10, 3.8))
for key, col, lab in [("wm", INDIGO, "cue hold (WM)"), ("stc", GREEN, "credit tag")]:
    m, lo, hi = _mci(r[key]); axA.plot(taus, m, "-o", color=col, lw=1.8, ms=4, label=lab)
    axA.fill_between(taus, lo, hi, color=col, alpha=0.15)
axA.axhline(0.75, ls="--", color=GREY, lw=1.0); axA.set_xscale("log")
axA.set_xticks(taus); axA.set_xticklabels([f"{t:g}" for t in taus])
axA.set_xlabel(r"retention $\tau_{leak}$ (s)"); axA.set_ylabel("task performance")
axA.set_ylim(0.45, 1.02); axA.set_title("(a) per-role requirement", fontsize=10)
axA.legend(fontsize=8, loc="lower right", frameon=False); _clean(axA)
m, lo, hi = _mci(r["one_shared"])
axB.axvspan(1.3, 6.0, color=GREEN, alpha=0.07)
axB.plot(taus, m, "-o", color=GREEN, lw=1.8, ms=4, label="one device, both roles")
axB.fill_between(taus, lo, hi, color=GREEN, alpha=0.15)
axB.axhline(0.75, ls="--", color=GREY, lw=1.0); axB.axhline(0.5, ls=":", color=GREY, lw=1.0)
axB.set_xscale("log"); axB.set_xticks(taus); axB.set_xticklabels([f"{t:g}" for t in taus])
axB.set_xlabel(r"retention $\tau_{leak}$ (s)"); axB.set_ylabel("joint-task performance")
axB.set_ylim(0.45, 1.02); axB.set_title("(b) single substrate, both roles", fontsize=10)
axB.legend(fontsize=8, loc="lower right", frameon=False); _clean(axB)
plt.show()

In [ ]:
r = paths.load_result("exp23_device_td.npy")
L = np.asarray(r["L_grid"], float); crit = r.get("crit", 0.75)
series = [("reinforce", INDIGO, "policy gradient (REINFORCE)"),
          ("td_actor_critic", GREEN, "device-native TD actor-critic"),
          ("td_no_homeo", GOLD, "TD, no eligibility homeostasis"),
          ("no_trace", GREY, "no-trace (chance)")]
def _mci(final, sc):
    m = np.array([np.asarray(final[(sc, l)]).mean() for l in r["L_grid"]])
    lo = np.array([np.percentile(np.asarray(final[(sc, l)]), 2.5) for l in r["L_grid"]])
    hi = np.array([np.percentile(np.asarray(final[(sc, l)]), 97.5) for l in r["L_grid"]])
    return m, lo, hi
fig, ax = plt.subplots(figsize=(6.6, 4.0))
for sc, col, lab in series:
    if (sc, r["L_grid"][0]) not in r["final"]:
        continue
    m, lo, hi = _mci(r["final"], sc)
    ax.plot(L, m, "-o", color=col, lw=1.7, ms=4, label=lab); ax.fill_between(L, lo, hi, color=col, alpha=0.13)
ax.axhline(crit, ls="--", color=GREY, lw=1.0)
ax.set_xlabel("corridor length $L$"); ax.set_ylabel("final goal-reach rate")
ax.set_xticks(L); ax.set_ylim(0, 1.03)
ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2, frameon=False)
ax.set_title("Device-native TD tracks but does not beat REINFORCE"); _clean(ax); plt.show()

In [ ]:
r = paths.load_result("exp21_beta_sensitivity.npy")
betas = list(r["betas"])                 # [1.0, 0.85, 0.54]
taus = np.asarray(r["taus_dmax"], float) # [1, 5, 10, 20]
dmax = r["dmax"]                          # {beta: {"dmax":[...per tau...], "k":..., "r2":...}}
bcol = [GREEN, INDIGO, GOLD]
fig, (axA, axB) = plt.subplots(1, 2, figsize=(9.6, 3.6))
# (a) D_max vs tau for each beta: the D_max ~ k*tau retention law, ~unchanged across dispersion
for b, c in zip(betas, bcol):
    dd = np.asarray(dmax[b]["dmax"], float); k = dmax[b]["k"]
    axA.plot(taus, dd, "-o", color=c, lw=1.7, ms=5, label=rf"$\beta={b:g}$  ($k={k:.1f}$)")
axA.set_xlabel(r"retention $\tau_{leak}$ (s)"); axA.set_ylabel(r"$D_{max}$ (s)")
axA.set_title("(a) retention law across dispersion", fontsize=10)
axA.legend(fontsize=8, frameon=False); _clean(axA)
# (b) the exp19 multi-timescale payoff (homeo vs best-single) across beta
exp19 = r["exp19"]; x = np.arange(len(betas)); w = 0.38
homeo = [exp19[b]["homeo"] for b in betas]; best = [exp19[b]["best"] for b in betas]
axB.bar(x - w / 2, homeo, w, color=GREEN, label="hetero + homeostasis")
axB.bar(x + w / 2, best, w, color=GREY, label="best single tau")
axB.set_xticks(x); axB.set_xticklabels([rf"$\beta={b:g}$" for b in betas])
axB.set_ylabel("overall reward rate"); axB.set_ylim(0, 1.03)
axB.set_title("(b) multi-timescale payoff vs dispersion", fontsize=10)
axB.legend(fontsize=8, frameon=False); _clean(axB)
plt.show()
print("D_max slope k across beta:", {b: round(dmax[b]["k"], 2) for b in betas})
print(r.get("verdict", ""))

In [ ]:
# Uncomment to launch the full sweeps as subprocesses (streamed; heavy -- minutes each):
# import subprocess, sys
# subprocess.run([sys.executable, "-m", "mrl_trace.extensions", "--exp22", "--exp23", "--full"])
print("see the markdown above for the full-scale commands")